In [99]:
import os
import torch
import numpy as np
import pandas as pd
import glob
from sklearn.neighbors import KDTree


In [100]:
def load_csv_files(root_dir, recursive, ignore_dirs, file_list):
    if file_list is not None:
        files = [os.path.join(root_dir, f) for f in file_list]
    elif recursive:
        # search recursively in subdirectories
        files = sorted(glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True))
    else:
        # search only in the root directory
        files = sorted(glob.glob(os.path.join(root_dir, "*.csv")))
    
    # filter out files in ignored directories
    if ignore_dirs:
        files = [f for f in files if not any(
            ig == os.path.basename(os.path.dirname(f)) for ig in ignore_dirs
        )]
    
    if not files:
        search_type = "recursively" if recursive else "in root directory"
        raise FileNotFoundError(f"No CSVs found {search_type} in {root_dir}")
    
    return files

def clean_dataset(df: pd.DataFrame, path: str) -> pd.DataFrame:
    """
    Clean the dataset by removing rows with NaN values in 'x-avg' or 'y-avg' columns.
    """

    if not {"time-rel-seconds", "x-avg", "y-avg"}.issubset(df.columns):
        raise ValueError(f"{path} must have columns: time-rel-seconds,x-avg,y-avg")

    df = df.sort_values("time-rel-seconds").reset_index(drop=True)

    df_cleaned = df.dropna().reset_index(drop=True)
    return df_cleaned

In [101]:
from torch_geometric.data import Data, HeteroData
kt = 2
ks = 3
def load_one(path: str, window_indices: slice) -> HeteroData:
    """
    Basic graph.
    V: time, x, y, pupil-left, pupil-right
    E-temporal: v, u not more than kt time-steps distant
    E-spatial: kNN (k:=ks)
    No edge weights yet.
    """
    # kt = kt
    # ks = ks

    df = pd.read_csv(path)

    df = clean_dataset(df, path)
    df = df.loc[window_indices, ["time-rel-seconds", "x-avg", "y-avg", "pupil-size-left-avg", "pupil-size-right-avg"]]
    n = len(df)
    print("shape", df.shape)

    #### node features matrix X
    X = torch.tensor(df[["time-rel-seconds", "x-avg", "y-avg", "pupil-size-left-avg", "pupil-size-right-avg"]].values, dtype=torch.float32)
    
    #### creating TEMPORAL edge_index matrix
    idx = torch.arange(n)          # [0, 1, ..., n-1]

    # relative offsets: -k..-1 and 1..k  (both directions)
    rel = torch.arange(-kt, kt + 1)
    rel = rel[rel != 0]            # drop 0

    # all candidate (src, dst) pairs before boundary check
    src = idx.repeat_interleave(len(rel))         # shape [n * (2k)]
    dst = (idx.view(-1, 1) + rel.view(1, -1)).reshape(-1)

    # mask out invalid indices (outside [0, n-1])
    valid = (dst >= 0) & (dst < n)
    src = src[valid]
    dst = dst[valid]

    edge_index_temporal = torch.stack([src, dst], dim=0)  # [2, E]

    #### creating SPATIAL edge_index matrix
    tree = KDTree(X[:, 1:3].numpy())  # use (x, y) for spatial neighbors
    dist, idx = tree.query(X[:, 1:3].numpy(), k=ks + 1)  # +1 to exclude self, idx shape: [N, k+1], idx[i, 0] == i (self)
    neighbors = idx[:, 1:].reshape(-1)          # drop self, flatten
    src = np.repeat(np.arange(n), ks)            # each node repeated k times

    edge_index_spatial = torch.tensor(
        np.vstack([np.concatenate([src, neighbors]),
                np.concatenate([neighbors, src])]),
        dtype=torch.long,
    )

    data = HeteroData()
    data["node"].x = X
    data["node"].num_nodes = n
    data["node", "temporal", "node"].edge_index = edge_index_temporal
    data["node", "spatial", "node"].edge_index = edge_index_spatial

    return data


In [102]:

root_dir = "."
recursive = True
ignore_dirs = []
file_list = None

files = load_csv_files(root_dir, recursive, ignore_dirs, file_list)

print(files[0])
g = load_one(files[0], slice(0, 100))
g

./data/processed/cog-load-mini/s_001.csv
shape (101, 5)


HeteroData(
  node={
    x=[101, 5],
    num_nodes=101,
  },
  (node, temporal, node)={ edge_index=[2, 398] },
  (node, spatial, node)={ edge_index=[2, 606] }
)

In [103]:
g["node"].x

tensor([[0.0000e+00, 9.7800e+02, 6.0000e+02, 3.3888e+00, 3.7185e+00],
        [1.6334e-02, 9.8600e+02, 6.0000e+02, 3.3874e+00, 3.7150e+00],
        [3.2982e-02, 9.8800e+02, 5.9600e+02, 3.3802e+00, 3.6976e+00],
        [4.9326e-02, 9.8450e+02, 6.0050e+02, 3.3650e+00, 3.6960e+00],
        [6.6191e-02, 9.8100e+02, 6.0500e+02, 3.3475e+00, 3.6877e+00],
        [8.3112e-02, 1.0100e+03, 5.9600e+02, 3.3480e+00, 3.6778e+00],
        [9.9196e-02, 1.0680e+03, 6.0700e+02, 3.3559e+00, 3.6693e+00],
        [1.1589e-01, 1.0940e+03, 6.0600e+02, 3.3703e+00, 3.6546e+00],
        [1.3281e-01, 1.1020e+03, 5.9100e+02, 3.3709e+00, 3.6463e+00],
        [1.4935e-01, 1.1020e+03, 5.9800e+02, 3.3655e+00, 3.6471e+00],
        [1.6583e-01, 1.0960e+03, 5.9500e+02, 3.3646e+00, 3.6504e+00],
        [1.8267e-01, 1.1020e+03, 5.9600e+02, 3.3651e+00, 3.6531e+00],
        [1.9905e-01, 1.1000e+03, 5.9900e+02, 3.3688e+00, 3.6481e+00],
        [2.1569e-01, 1.1020e+03, 5.9400e+02, 3.3660e+00, 3.6468e+00],
        [2.3272e-01,

In [105]:
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import FancyArrowPatch
%matplotlib qt
# Extract first 100 nodes
num_nodes = min(100, g["node"].num_nodes)
X_subset = g["node"].x[:num_nodes]

# Filter edges to only include those within first 100 nodes
temporal_edges = g["node", "temporal", "node"].edge_index
spatial_edges = g["node", "spatial", "node"].edge_index

# Filter temporal edges
temporal_mask = (temporal_edges[0] < num_nodes) & (temporal_edges[1] < num_nodes)
temporal_edges_filtered = temporal_edges[:, temporal_mask]

# Filter spatial edges
spatial_mask = (spatial_edges[0] < num_nodes) & (spatial_edges[1] < num_nodes)
spatial_edges_filtered = spatial_edges[:, spatial_mask]

# Create networkx graph
G = nx.Graph()
G.add_nodes_from(range(num_nodes))

# Add edges with types
for i in range(temporal_edges_filtered.shape[1]):
    src, dst = temporal_edges_filtered[0, i].item(), temporal_edges_filtered[1, i].item()
    G.add_edge(src, dst, edge_type='temporal')

for i in range(spatial_edges_filtered.shape[1]):
    src, dst = spatial_edges_filtered[0, i].item(), spatial_edges_filtered[1, i].item()
    G.add_edge(src, dst, edge_type='spatial')

# Create figure
fig, ax = plt.subplots(figsize=(16, 16))

# Use x,y coordinates from node features as positions
pos = {}
for i in range(num_nodes):
    x = X_subset[i, 1].item()  # x coordinate at index 1
    y = X_subset[i, 2].item()  # y coordinate at index 2
    pos[i] = (x, y)

# Draw temporal edges (blue) with arrows from smaller to bigger index
temporal_edge_list = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'temporal']
for u, v in temporal_edge_list:
    # Ensure arrow goes from smaller to bigger index
    start, end = (u, v) if u < v else (v, u)
    alpha = 0.2 + (start / num_nodes) * 0.7  # Alpha from 0.2 to 0.9 based on source node index
    
    # Create arrow patch
    arrow = FancyArrowPatch(pos[start], pos[end],
                           arrowstyle='->', mutation_scale=10, 
                           color='blue', alpha=alpha, linewidth=0.5, zorder=1)
    ax.add_patch(arrow)

# Draw spatial edges (red)
spatial_edge_list = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'spatial']
for u, v in spatial_edge_list:
    alpha = 0.2 + (u / num_nodes) * 0.7  # Alpha from 0.2 to 0.9 based on source node index
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], 
            'r-', alpha=alpha, linewidth=0.5, zorder=1)

# Draw nodes (green) with varying alpha
for node in range(num_nodes):
    alpha = 0.2 + (node / num_nodes) * 0.7
    ax.scatter(pos[node][0], pos[node][1], c='green', s=50, alpha=alpha, zorder=2)

ax.set_title(f'Graph Visualization (First {num_nodes} Nodes)', fontsize=16)
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')
plt.legend()
plt.tight_layout()

# Display in separate window
plt.show()

print(f"Visualized {num_nodes} nodes")
print(f"Temporal edges: {len(temporal_edge_list)}")
print(f"Spatial edges: {len(spatial_edge_list)}")


Visualized 100 nodes
Temporal edges: 115
Spatial edges (undirected): 201

Spatial degree per node (first 10):
  Node 0: 4 edges
  Node 1: 5 edges
  Node 2: 4 edges
  Node 3: 5 edges
  Node 4: 3 edges
  Node 5: 3 edges
  Node 6: 3 edges
  Node 7: 4 edges
  Node 8: 4 edges
  Node 9: 6 edges


/var/folders/h5/svdfmb2x5xx25t58fnytwfvc0000gn/T/ipykernel_11214/1877290952.py:74: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
